# E08B — Stronger-Model Diagnostic (A2-GPT5mini vs A2-Qwen)

Reconstruction-v2. Matched comparison: the ONLY variable changed from E07 is the model (`qwen2.5:7b-instruct-ctx16k` -> `openai/gpt-5-mini`). Manifest, retrieved context, prompt (`classification_prompt_v1`), output schema, parser, and evidence validator are all held fixed. This notebook loads already-computed result files only -- no new LLM calls are made here.

In [1]:
import json, sys
from pathlib import Path
REPO = Path.cwd().resolve().parents[1] if (Path.cwd() / 'pipeline').exists() is False else Path.cwd()
import os
REPO = Path(os.environ.get('NDATRACE_REPO', Path.cwd()))
E08B = REPO / 'experiments/E08B_stronger_model_diagnostic'
E07 = REPO / 'experiments/E07_standard_rag'
def load(p):
    with open(p) as f:
        return json.load(f)
print('repo:', REPO)

repo: /Users/asmitha/Documents/course/PE6201-EMERGING AI TECHNOLOGIES/Project/ndatrace


## 1. Research question

How much of A2 Standard RAG's remaining failure is attributable to the base model rather than the retrieval architecture? E08 found the largest failure bucket (49.2% overall, 65.5% for Contradiction) was `MODEL_REASONING_LIMITED` with no identified information-acquisition fix. This diagnostic isolates model capability by substituting `openai/gpt-5-mini` for `qwen2.5:7b-instruct-ctx16k` and nothing else.

## 2. Frozen A2 setup

In [2]:
summary_md = (E08B / 'summary.md').read_text()
print(summary_md[:2000])

# E08B — Stronger-Model Diagnostic — STAGE A (design/audit + budget gate only)

**No GPT-5 mini call has been made. No Qwen call has been made. `retrieval_v1` and
`classification_prompt_v1` are read-only inputs.** This document audits the current GPT-5 mini
provider/pricing configuration, verifies the frozen A2 input artifact's integrity, reads the
real reconstruction-v2 spend ledger, applies the existing pre-run budget gate, projects the
150-case cost using real historical output-token data, and proposes the full matched-comparison,
E08-bucket-recovery, and statistical analysis plan for Stage B.

## 1. Research question

"How much of A2 Standard RAG's remaining failure is attributable to the base model rather than
the retrieval architecture?" The only change from E07: `qwen2.5:7b-instruct-ctx16k` →
`openai/gpt-5-mini`. Everything else — manifest, retrieved context, prompt, schema, parser,
evidence validator — held fixed. This is model-capability isolation, not prompt tuning: no
GPT-sp

## 3. Calibration and revised cost forecast

In [3]:
calib = load(E08B / 'results/calibration_8case.json')
print('timeout:', calib['request_timeout_seconds'])
print('strict/recovered/invalid:', calib['strict_parse_count'], calib['recovered_parse_count'], calib['invalid_parse_count'])
print('revised 150-case projection:', json.dumps(calib['revised_150_case_projection'], indent=2))

timeout: 30
strict/recovered/invalid: 8 0 0
revised 150-case projection: {
  "mean_cost_per_case_usd": 0.0017233125,
  "max_cost_per_case_usd": 0.003106,
  "mean_output_tokens": 698.375,
  "max_output_tokens": 1408,
  "projected_150_case_central_usd": 0.258496875,
  "projected_150_case_conservative_usd": 0.4659,
  "gate_result": "OK: projected total $0.5974 <= allowed $3.7500 (planning budget $5.00 - reserve $1.25)",
  "gate_allowed": true
}


## 4. Budget gate (pre- and post-full-run)

In [4]:
wall = load(E08B / 'results/run_E08B_A2_gpt5mini_train_wall_seconds.json')
print(json.dumps(wall, indent=2))

{
  "total_wall_seconds": 1112.1559341250686,
  "n_cases": 150,
  "run_id": "3604cb3f5ac7",
  "run_spend_usd": 0.2556927500000002,
  "pre_run_ledger_spend_usd": 0.1315464500000001,
  "post_run_ledger_spend_usd": 0.3872392
}


## 5. Full-run verification

All 150 cases completed, 0 errors, 0 timeouts (see run log). Retrieval was NOT re-run -- the frozen `TRAIN_ARCH_v1_RETRIEVED_retrieval_v1.json` artifact was read directly and its integrity verified before the run started.

In [5]:
rows = [json.loads(l) for l in open(E08B / 'results/run_E08B_A2_gpt5mini_train_cases.jsonl')]
print('n_cases:', len(rows))
print('errors:', sum(1 for r in rows if r.get('error_type')))
from collections import Counter
print(Counter(r['parse_status'] for r in rows))

n_cases: 150
errors: 0
Counter({'strict': 150})


## 6. GPT classification/evidence/joint metrics

In [6]:
gpt_summary = load(E08B / 'results/run_E08B_A2_gpt5mini_train.json')
print(json.dumps(gpt_summary['classification'], indent=2, default=str))
print(json.dumps(gpt_summary['evidence'], indent=2))
print(json.dumps(gpt_summary['joint'], indent=2))

{
  "accuracy": 0.7866666666666666,
  "macro_f1": 0.7868620987914067,
  "per_class_recall": {
    "Entailment": 0.82,
    "Contradiction": 0.76,
    "NotMentioned": 0.78
  },
  "contradiction_recall": 0.76,
  "contradiction_recall_ci95": [
    0.6258705062480764,
    0.8570274759812778
  ],
  "contradiction_n": 50,
  "confusion_matrix": {
    "labels": [
      "Entailment",
      "Contradiction",
      "NotMentioned"
    ],
    "matrix": [
      [
        41,
        3,
        6
      ],
      [
        10,
        38,
        2
      ],
      [
        4,
        7,
        39
      ]
    ]
  }
}
{
  "evidence_bearing_n": 100,
  "evidence_recall": 0.86,
  "evidence_precision": 0.8349514563106796,
  "correct_label_bad_evidence_count": 4,
  "wrong_label_source_valid_evidence_count": 20,
  "paraphrased_evidence_count": 6
}
{
  "overall": 0.74,
  "by_class": {
    "Entailment": 0.74,
    "Contradiction": 0.7,
    "NotMentioned": 0.78
  }
}


## 7. Qwen vs GPT matched metric table

In [7]:
cmp = load(E08B / 'results/qwen_vs_gpt_paired_comparison.json')
import pandas as pd
df = pd.DataFrame(cmp['metric_table']).T
df

,Qwen,GPT-5-mini,delta_GPT_minus_Qwen
accuracy,0.433333,0.786667,0.353333
macro_f1,0.430886,0.786862,0.355976
entailment_recall,0.340000,0.820000,0.480000
contradiction_recall,0.420000,0.760000,0.340000
notmentioned_recall,0.540000,0.780000,0.240000
joint_overall,0.333333,0.740000,0.406667
joint_entailment,0.160000,0.740000,0.580000
joint_contradiction,0.300000,0.700000,0.400000
joint_notmentioned,0.540000,0.780000,0.240000
evidence_recall,0.290000,0.860000,0.570000


## 8. Confusion matrices

In [8]:
qwen_summary = load(E07 / 'results/run_E07_A2_train.json')
print('Qwen confusion matrix (rows=gold, cols=pred):')
print(pd.DataFrame(qwen_summary['classification']['confusion_matrix']['matrix'],
                    index=qwen_summary['classification']['confusion_matrix']['labels'],
                    columns=qwen_summary['classification']['confusion_matrix']['labels']))
print('\nGPT-5-mini confusion matrix:')
print(pd.DataFrame(gpt_summary['classification']['confusion_matrix']['matrix'],
                    index=gpt_summary['classification']['confusion_matrix']['labels'],
                    columns=gpt_summary['classification']['confusion_matrix']['labels']))

Qwen confusion matrix (rows=gold, cols=pred):
               Entailment  Contradiction  NotMentioned
Entailment             17             17            16
Contradiction           3             21            26
NotMentioned           12             11            27

GPT-5-mini confusion matrix:
               Entailment  Contradiction  NotMentioned
Entailment             41              3             6
Contradiction          10             38             2
NotMentioned            4              7            39


## 9. Contradiction-specific comparison

In [9]:
print('Qwen Contradiction recall:', qwen_summary['classification']['contradiction_recall'],
      qwen_summary['classification']['contradiction_recall_ci95'])
print('GPT Contradiction recall:', gpt_summary['classification']['contradiction_recall'],
      gpt_summary['classification']['contradiction_recall_ci95'])
print('Transitions (Contradiction):', cmp['transitions_contradiction'])
print('McNemar (Contradiction):', cmp['mcnemar_contradiction'])

Qwen Contradiction recall: 0.42 [0.2937479723456693, 0.5576680331222217]
GPT Contradiction recall: 0.76 [0.6258705062480764, 0.8570274759812778]
Transitions (Contradiction): {'qwen_wrong_gpt_correct': 24, 'qwen_correct_gpt_wrong': 7, 'both_correct': 14, 'both_wrong': 5}
McNemar (Contradiction): {'b_qwen_only_correct': 7, 'c_gpt_only_correct': 24, 'n_discordant': 31, 'p_value': 0.003326892852783203, 'significant_at_0.05': True}


## 10. Evidence metrics

In [10]:
print('Qwen evidence:', json.dumps(qwen_summary['evidence'], indent=2))
print('GPT evidence:', json.dumps(gpt_summary['evidence'], indent=2))

Qwen evidence: {
  "evidence_bearing_n": 100,
  "evidence_recall": 0.29,
  "evidence_precision": 0.35802469135802467,
  "correct_label_bad_evidence_count": 14,
  "wrong_label_valid_evidence_count": 54,
  "paraphrased_evidence_count": 32
}
GPT evidence: {
  "evidence_bearing_n": 100,
  "evidence_recall": 0.86,
  "evidence_precision": 0.8349514563106796,
  "correct_label_bad_evidence_count": 4,
  "wrong_label_source_valid_evidence_count": 20,
  "paraphrased_evidence_count": 6
}


## 11. Joint metric

Qwen joint success (all 150 cases, from E07's own `rag_failure_analysis.csv`, `joint_success` column) = 50/150 = 33.3%. GPT joint success (all 150) = 111/150 = 74.0%. Both use the identical joint-success semantics (`scripts/analyze_e07_standard_rag.py`'s `joint_success()`, reused verbatim in `scripts/analyze_e08b_stronger_model.py`) -- correct label AND (gold-evidence-overlap >= 0.5 for Entailment/Contradiction, or no evidence claimed for NotMentioned).

In [11]:
print('Qwen joint (150-case, from E07 analysis):', qwen_summary['joint'])
print('GPT joint (150-case):', gpt_summary['joint'])
print('Joint transitions (FULL 150 matched cases):', cmp['transitions_joint_full150'])
jt = cmp['transitions_joint_full150']
assert sum(jt.values()) == 150
assert jt['both_correct'] + jt['qwen_correct_gpt_wrong'] == 50  # Qwen joint marginal
assert jt['both_correct'] + jt['qwen_wrong_gpt_correct'] == 111  # GPT joint marginal
print('marginals verified: Qwen=50, GPT=111, total=150')

Qwen joint (150-case, from E07 analysis): {'overall': 0.3333333333333333, 'by_class': {'Entailment': 0.16, 'Contradiction': 0.3, 'NotMentioned': 0.54}}
GPT joint (150-case): {'overall': 0.74, 'by_class': {'Entailment': 0.74, 'Contradiction': 0.7, 'NotMentioned': 0.78}}
Joint transitions (FULL 150 matched cases): {'qwen_wrong_gpt_correct': 75, 'qwen_correct_gpt_wrong': 14, 'both_correct': 36, 'both_wrong': 25}
marginals verified: Qwen=50, GPT=111, total=150


## 12. Case-level transitions

In [12]:
print('Overall:', cmp['transitions_overall'])
print('Contradiction:', cmp['transitions_contradiction'])

Overall: {'qwen_wrong_gpt_correct': 69, 'qwen_correct_gpt_wrong': 16, 'both_correct': 49, 'both_wrong': 16}
Contradiction: {'qwen_wrong_gpt_correct': 24, 'qwen_correct_gpt_wrong': 7, 'both_correct': 14, 'both_wrong': 5}


## 13. E08 failure-bucket recovery (headline diagnostic)

In [13]:
bucket = load(E08B / 'results/e08_bucket_recovery.json')
print(json.dumps(bucket['e08_bucket_recovery'], indent=2, default=str))

{
  "model_reasoning_limited": {
    "n": 31,
    "gpt_classification_correct_count": 30,
    "gpt_classification_correct_pct": 96.7741935483871,
    "gpt_joint_success_count": 28,
    "gpt_joint_success_pct": 90.32258064516128,
    "qwen_to_gpt_fixes": 30,
    "qwen_to_gpt_regressions": 0,
    "note": "All cases in this bucket are, by E08's own construction, Qwen ERRORS -- 'fixes' and 'gpt_classification_correct_count' are therefore the same number; regressions are structurally impossible within this bucket definition."
  },
  "agentically_fixable": {
    "n": 26,
    "gpt_classification_correct_count": 19,
    "gpt_classification_correct_pct": 73.07692307692307,
    "gpt_joint_success_count": 18,
    "gpt_joint_success_pct": 69.23076923076923,
    "qwen_to_gpt_fixes": 19,
    "qwen_to_gpt_regressions": 0,
    "note": "All cases in this bucket are, by E08's own construction, Qwen ERRORS -- 'fixes' and 'gpt_classification_correct_count' are therefore the same number; regressions are st

## 14. 54-case wrong-evidence subgroup

In [14]:
print(json.dumps(bucket['wrong_evidence_subgroup_54cases'], indent=2))

{
  "group_a_gold_overlap_4cases": {
    "n": 4,
    "gpt_classification_correct_count": 3,
    "gpt_classification_correct_pct": 75.0,
    "gpt_joint_success_count": 3,
    "gpt_source_valid_evidence_count": 4,
    "gpt_gold_overlap_evidence_count": 4,
    "gpt_still_wrong_and_source_valid_count": 1,
    "gpt_still_wrong_and_non_gold_evidence_count": 0
  },
  "group_b_no_gold_overlap_50cases": {
    "n": 50,
    "gpt_classification_correct_count": 39,
    "gpt_classification_correct_pct": 78.0,
    "gpt_joint_success_count": 35,
    "gpt_source_valid_evidence_count": 48,
    "gpt_gold_overlap_evidence_count": 41,
    "gpt_still_wrong_and_source_valid_count": 11,
    "gpt_still_wrong_and_non_gold_evidence_count": 7
  },
  "combined_54cases": {
    "n": 54,
    "gpt_classification_correct_count": 42,
    "gpt_classification_correct_pct": 77.77777777777779,
    "gpt_joint_success_count": 38,
    "gpt_source_valid_evidence_count": 52,
    "gpt_gold_overlap_evidence_count": 45,
    "gpt_st

## 15. Static-six (reranker-limited) analysis

Gold evidence was excluded from the final top-5 by the frozen retrieval pipeline for all 6 cases. A GPT failure here is NOT evidence of poor reasoning.

In [15]:
print(json.dumps(bucket['static_six_reranker_limited'], indent=2, default=str))

{
  "case_ids": [
    "train::160::nda-10",
    "train::247::nda-10",
    "train::353::nda-10",
    "train::379::nda-10",
    "train::438::nda-2",
    "train::518::nda-10"
  ],
  "gpt_outcomes": [
    {
      "case_id": "train::160::nda-10",
      "gold_label": "Entailment",
      "gpt_predicted_label": "NotMentioned",
      "gpt_correct": false
    },
    {
      "case_id": "train::247::nda-10",
      "gold_label": "Entailment",
      "gpt_predicted_label": "NotMentioned",
      "gpt_correct": false
    },
    {
      "case_id": "train::353::nda-10",
      "gold_label": "Entailment",
      "gpt_predicted_label": "NotMentioned",
      "gpt_correct": false
    },
    {
      "case_id": "train::379::nda-10",
      "gold_label": "Entailment",
      "gpt_predicted_label": "NotMentioned",
      "gpt_correct": false
    },
    {
      "case_id": "train::438::nda-2",
      "gold_label": "Contradiction",
      "gpt_predicted_label": "Contradiction",
      "gpt_correct": true
    },
    {
     

## 16. Paired statistics (McNemar + bootstrap)

In [16]:
print('McNemar overall:', cmp['mcnemar_overall'])
print('McNemar Contradiction:', cmp['mcnemar_contradiction'])
print('Bootstrap accuracy delta:', cmp['bootstrap_accuracy_delta_GPT_minus_Qwen'])
print('Bootstrap macro-F1 delta:', cmp['bootstrap_macro_f1_delta_GPT_minus_Qwen'])
print('Bootstrap Contradiction recall delta:', cmp['bootstrap_contradiction_recall_delta_GPT_minus_Qwen'])
print('Bootstrap joint delta (full 150):', cmp['bootstrap_joint_delta_GPT_minus_Qwen_full150'])

McNemar overall: {'b_qwen_only_correct': 16, 'c_gpt_only_correct': 69, 'n_discordant': 85, 'p_value': 5.240266679443081e-09, 'significant_at_0.05': True}
McNemar Contradiction: {'b_qwen_only_correct': 7, 'c_gpt_only_correct': 24, 'n_discordant': 31, 'p_value': 0.003326892852783203, 'significant_at_0.05': True}
Bootstrap accuracy delta: {'point_estimate': 0.3533333333333333, 'ci95_low': 0.2466666666666667, 'ci95_high': 0.45999999999999996}
Bootstrap macro-F1 delta: {'point_estimate': 0.3559760710617635, 'ci95_low': 0.24979579740842273, 'ci95_high': 0.4616923023402246}
Bootstrap Contradiction recall delta: {'point_estimate': 0.34, 'ci95_low': 0.14, 'ci95_high': 0.5399999999999999}
Bootstrap joint delta (full 150): {'point_estimate': 0.4066666666666667, 'ci95_low': 0.30000000000000004, 'ci95_high': 0.5066666666666667}


## 17. Latency

In [17]:
print('Qwen generation latency (ms):', qwen_summary['generation_latency_ms'])
print('GPT generation latency (ms):', gpt_summary['generation_latency_ms'])

Qwen generation latency (ms): {'mean': 7266.066341951179, 'median': 6106.188000005204, 'p90': 11730.843666009605, 'max': 19851.623042020947}
GPT generation latency (ms): {'mean': 7411.432088869624, 'median': 6684.958771045785, 'p90': 12549.430458107963, 'max': 16792.086415924132}


## 18. Actual cost

In [18]:
print('GPT cost summary:', json.dumps(gpt_summary['cost_usd'], indent=2))
print('Qwen total cost: $0.00 (local Ollama)')

GPT cost summary: {
  "total": 0.25569275,
  "mean_per_case": 0.0017046183333333336,
  "median_per_case": 0.00162725,
  "p90_per_case": 0.002783,
  "max_per_case": 0.0039265
}
Qwen total cost: $0.00 (local Ollama)


## 19. Cost-effectiveness

Additional GPT spend per additional correct classification / joint-success case vs Qwen.

In [19]:
# Exact counts from the transitions tables (not rounded from aggregate percentages).
t = cmp['transitions_overall']
qwen_correct_n = t['both_correct'] + t['qwen_correct_gpt_wrong']
gpt_correct_n = t['both_correct'] + t['qwen_wrong_gpt_correct']
additional_correct = gpt_correct_n - qwen_correct_n
gpt_total_cost = gpt_summary['cost_usd']['total']
print(f'Qwen correct: {qwen_correct_n}, GPT correct: {gpt_correct_n}, additional: {additional_correct}')
if additional_correct > 0:
    print(f'Additional spend per additional correct case: ${gpt_total_cost/additional_correct:.5f}')
else:
    print('Not meaningful -- denominator <= 0')

jt = cmp['transitions_joint_full150']
qwen_joint_n = jt['both_correct'] + jt['qwen_correct_gpt_wrong']
gpt_joint_n = jt['both_correct'] + jt['qwen_wrong_gpt_correct']
additional_joint = gpt_joint_n - qwen_joint_n
print(f'Qwen joint: {qwen_joint_n}, GPT joint: {gpt_joint_n}, additional: {additional_joint}')
assert qwen_joint_n == 50 and gpt_joint_n == 111 and additional_joint == 61
if additional_joint > 0:
    print(f'Additional spend per additional joint-success case: ${gpt_total_cost/additional_joint:.5f}')
else:
    print('Not meaningful -- denominator <= 0')

Qwen correct: 65, GPT correct: 118, additional: 53
Additional spend per additional correct case: $0.00482
Qwen joint: 50, GPT joint: 111, additional: 61
Additional spend per additional joint-success case: $0.00419


## 20. Implication for A3 (agent) decision

**Diagnostic conclusion: A — STRONGER MODEL LARGELY SOLVES THE REASONING BOTTLENECK.**

GPT-5-mini recovers 96.8% (30/31) of E08's `MODEL_REASONING_LIMITED` cases and 73.1% (19/26) of `AGENTICALLY_FIXABLE` cases without any retrieval or agent change — the latter suggests E08 overestimated how much of that bucket genuinely required dynamic information-seeking versus simply needing a more capable reasoner over the same evidence. Only 1/6 `STATIC_PIPELINE_FIXABLE` cases resolve (expected — gold evidence was absent from context for both models). Overall accuracy: Qwen 43.3% -> GPT 78.7% (McNemar p=5.2e-9); Contradiction recall: 42.0% -> 76.0% (p=0.0033); joint success (full 150 cases): 33.3% -> 74.0% (+61 cases, McNemar-equivalent transition table above). This is not a final production/purchasing decision — no GPT-specific prompt engineering was attempted, and A3 has not been built.

**This does NOT mean agents are unnecessary.** The static-six retrieval/filtering-limited cases and GPT's own residual failures (16/150 classification, 14/150 joint, still wrong) have not yet been analyzed for agentic recoverability. **E09 should reassess agent justification using GPT's residual failures, not Qwen's** — A3 must not be designed to solve a reasoning weakness that was specific to the weaker model and has now mostly disappeared. Any remaining A3 justification must come from failures where (1) information is genuinely missing or incomplete in the retrieved context, (2) a case-dependent action could plausibly obtain it, and (3) GPT-5-mini still fails without that action -- a narrower and smaller target than E08's original Qwen-based bucket.